In [ ]:
import kagglehub

# Download dataset
dataset_path = kagglehub.dataset_download("miadul/animal-image-classification-5-species")

print("Dataset downloaded to:", dataset_path)

### Step 1: Dataset Directory Setup
Pehle hum check karein ge k dataset ka folder structure kaisa hai. Humein data ko training aur validation split mein divide karna hoga Keras image_dataset_from_directory use kar ke.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dataset path set karein
DATA_DIR = dataset_path 

# Folder structure dekhein
print("Subfolders:", os.listdir(DATA_DIR))

### Step 2: Data Loading & Train/Validation Split
TensorFlow ka utility generator use kar ke data ko split karein ge bina extra storage lagaye.

In [ ]:
IMG_DIMS = (224, 224)
BATCH_SZ = 32

# Training set (80%)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=45,
    image_size=IMG_DIMS,
    batch_size=BATCH_SZ
)

In [ ]:
# Validation set (20%)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=45,
    image_size=IMG_DIMS,
    batch_size=BATCH_SZ
)

In [ ]:
# Classes check karein
class_list = train_dataset.class_names
print("Animal Classes:", class_list)
print("Total classes:", len(class_list))

### Step 3: Performance Optimization with Autotune
Dataset ko cache aur prefetch karein ge taake computation fast ho aur bottleneck na aaye.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(800).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

### Step 4: Data Augmentation & Transfer Learning
Yeh Multi-Class Classification hai is liye hum ResNet50 use kar rahe hain jo ke MobileNetV2 se alag architecture hai. Saath mein data augmentation bhi apply kar rahe hain taake model generalize kare.

In [ ]:
# Data Augmentation layers
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

In [ ]:
# ResNet50 base model load karein (ImageNet weights)
backbone = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

In [ ]:
# Backbone ko freeze karein
backbone.trainable = False

In [ ]:
# Custom classifier head (Multi-class)
classifier = tf.keras.Sequential([
    # Rescaling for ResNet50 (expects -1 to 1 range)
    tf.keras.layers.Rescaling(1./127.5, offset=-1),
    data_augmentation,
    backbone,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_list), activation='softmax')
])

### Step 5: Model Compilation & Training
Model compile karein aur train karein. Learning rate thoda kam rakha hai taake fine-tuning achi ho.

In [ ]:
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

classifier.summary()

# Early stopping
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True
)

# Learning rate scheduler
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7
)

# Training
training_history = classifier.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=12,
    callbacks=[stop_early, lr_scheduler]
)